# Day 3 — Churn Machine Learning Models

Train and compare **Logistic Regression**, **Random Forest**, and **XGBoost** on the Day 2 ML-ready data.

**Pipeline**
```
Day 2 processed X/y → Train 3 models → Evaluate → Select best → Save → Predict + Risk
```

No LTV / SHAP / API in this notebook (later days).

## 1. Setup and load Day 2 data

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from churn_model import (
    build_sample_predictions_table,
    evaluate_all_models,
    evaluate_model,
    get_test_customer_ids,
    load_churn_model,
    load_preprocessor,
    load_processed_splits,
    predict_churn,
    predict_from_raw_customer,
    risk_level,
    save_model,
    select_best_model,
    train_models,
)

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
X_train, X_test, y_train, y_test = load_processed_splits()
preprocessor = load_preprocessor()

print(f"Training features: {X_train.shape}")
print(f"Testing features:  {X_test.shape}")
print(f"Training target:   {y_train.shape}")
print(f"Testing target:    {y_test.shape}")
print(f"Train churn %: {y_train.mean()*100:.2f}")
print(f"Test churn %:  {y_test.mean()*100:.2f}")
print("\nPreprocessor type:", type(preprocessor).__name__)
print("\nWhy Accuracy alone is not enough:")
print("About 73.5% of customers do not churn. A model that always predicts No")
print("would look accurate but would miss almost every churner. So we also")
print("watch Recall, F1, and ROC-AUC for the churn class (Yes=1).")

## 2. What each model does

| Model | Purpose |
|-------|--------|
| **Logistic Regression** | Simple linear baseline; easy to understand |
| **Random Forest** | Many decision trees; captures non-linear patterns |
| **XGBoost** | Gradient-boosted trees; often strong on tabular data |

## 3. Train all three models

We train on **Day 2 processed training data** (preprocessor was fitted on train only — no test leakage).

In [ ]:
trained = train_models(X_train, y_train)
print("Trained models:", list(trained.keys()))

## 4. Evaluate each model

In [ ]:
def show_eval(name, model):
    m = evaluate_model(model, X_test, y_test)
    print(f"\n=== {name} ===")
    print(f"Accuracy:  {m['Accuracy']:.4f}")
    print(f"Precision: {m['Precision']:.4f}  (of predicted churners, how many truly churned)")
    print(f"Recall:    {m['Recall']:.4f}  (of actual churners, how many we caught)")
    print(f"F1 Score:  {m['F1 Score']:.4f}")
    print(f"ROC-AUC:   {m['ROC-AUC']:.4f}")
    print("Confusion Matrix [[TN, FP], [FN, TP]]:")
    print(np.array(m["Confusion Matrix"]))
    return m

m_lr = show_eval("Logistic Regression", trained["Logistic Regression"])
m_rf = show_eval("Random Forest", trained["Random Forest"])
m_xgb = show_eval("XGBoost", trained["XGBoost"])

## 5. Confusion matrix plots

In [ ]:
def plot_cm(cm, title, filename):
    labels = np.array([["TN", "FP"], ["FN", "TP"]])
    annot = np.array([[f"{labels[i,j]}\n{cm[i][j]}" for j in range(2)] for i in range(2)])
    plt.figure(figsize=(5, 4))
    sns.heatmap(np.array(cm), annot=annot, fmt="", cmap="Blues",
                xticklabels=["Pred No", "Pred Yes"],
                yticklabels=["Actual No", "Actual Yes"], cbar=False)
    plt.title(title)
    path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

plot_cm(m_lr["Confusion Matrix"], "Logistic Regression", "logistic_confusion_matrix.png")
plot_cm(m_rf["Confusion Matrix"], "Random Forest", "random_forest_confusion_matrix.png")
plot_cm(m_xgb["Confusion Matrix"], "XGBoost", "xgboost_confusion_matrix.png")

## 6. Model comparison and best-model selection

In [ ]:
comparison, details = evaluate_all_models(trained, X_test, y_test)
display(comparison)
comparison.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)

best_name, explanation = select_best_model(comparison)
best_model = trained[best_name]
best_metrics = details[best_name]
print("\nBest model:", best_name)
print(explanation)

## 7. Save best model

In [ ]:
metadata = {
    "best_model_name": best_name,
    "selection_explanation": explanation,
    "metrics": {k: best_metrics[k] for k in ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]},
    "comparison": comparison.to_dict(orient="records"),
    "risk_thresholds": {
        "low": "< 0.30",
        "medium": "0.30 to < 0.60",
        "high": ">= 0.60",
        "note": "Initial business thresholds; can be tuned later.",
    },
}
save_model(best_model, metadata)
print("Saved:", MODELS_DIR / "churn_model.pkl")
print("Saved:", MODELS_DIR / "churn_model_metadata.pkl")

## 8. Churn probability and risk levels

Initial thresholds (tunable later, not claimed as optimal):
- **Low:** 0.00–0.30
- **Medium:** 0.30–0.60
- **High:** 0.60–1.00

In [ ]:
all_preds = predict_churn(best_model, X_test)
proba = np.array(all_preds["probability"])
risks = pd.Series(all_preds["risk_level"])
risk_counts = risks.value_counts().reindex(["Low", "Medium", "High"]).fillna(0).astype(int)
print("Risk distribution on test set:")
print(risk_counts)
print(f"\nHigh-risk customers: {int(risk_counts['High'])} / {len(X_test)}")

plt.figure(figsize=(8, 5))
sns.histplot(proba, bins=30, kde=True, color="steelblue")
plt.axvline(0.30, color="orange", linestyle="--", label="Medium (0.30)")
plt.axvline(0.60, color="red", linestyle="--", label="High (0.60)")
plt.title(f"Predicted Churn Probability ({best_name})")
plt.xlabel("Churn probability")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "churn_probability_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(6, 4))
sns.barplot(x=risk_counts.index, y=risk_counts.values, hue=risk_counts.index,
            palette=["#2ca02c", "#ff7f0e", "#d62728"], legend=False)
plt.title("Risk Level Distribution (Test Set)")
plt.ylabel("Customers")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "risk_level_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Sample predictions (real test customers)

In [ ]:
id_test = get_test_customer_ids()
sample = build_sample_predictions_table(best_model, X_test, y_test, id_test, n=10)
display(sample)
sample.to_csv(REPORTS_DIR / "sample_predictions.csv", index=False)

## 10. Reload saved model (FastAPI will do this later)

In [ ]:
loaded_model = load_churn_model()
loaded_pre = load_preprocessor()
print("Model loaded:", type(loaded_model).__name__)
print("Preprocessor loaded:", type(loaded_pre).__name__)

demo = predict_churn(loaded_model, X_test.head(3))
print("Sample reload predictions:", demo)

cleaned = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "cleaned_telco.csv")
one = cleaned.iloc[[0]]
print("End-to-end prediction for", one["customerID"].iloc[0], ":")
print(predict_from_raw_customer(one, loaded_model, loaded_pre))

print("\nRisk helper examples:")
for p in [0.12, 0.45, 0.81]:
    print(f"  probability={p:.2f} → {risk_level(p)}")

## 11. Business interpretation

Run the cell below so the text uses **your actual metrics** from this notebook run.

In [ ]:
high_n = int(risk_counts["High"])
print("BUSINESS INTERPRETATION")
print("-----------------------")
print(f"Best model: {best_name}")
print(f"F1 Score:  {best_metrics['F1 Score']:.4f}")
print(f"Recall:    {best_metrics['Recall']:.4f}")
print(f"ROC-AUC:   {best_metrics['ROC-AUC']:.4f}")
print(f"Precision: {best_metrics['Precision']:.4f}")
print(f"Accuracy:  {best_metrics['Accuracy']:.4f}")
print(f"High-risk test customers (prob >= 0.60): {high_n} of {len(X_test)}")
print()
print("For a telecom company, the high-risk group is a practical starting list")
print("for retention campaigns (offers, contract upgrades, support outreach).")
print("Day 4 will add LTV and SHAP explanations on top of this model.")